In [1]:
#Statistical Image Analysis Steps
"""
================================================================================
 FULL 10-STAGE STATISTICAL DENOISING PIPELINE
================================================================================

"""
import os, string
import numpy as np
import cv2
from scipy import ndimage, stats
from skimage.util import img_as_float, img_as_ubyte, random_noise
from skimage.restoration import estimate_sigma, denoise_nl_means
from skimage.measure import shannon_entropy
from skimage.metrics import mean_squared_error as MSE
from skimage.metrics import peak_signal_noise_ratio as PSNR
from skimage.metrics import structural_similarity as SSIM
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

CLEAN = "org_img.jpg"
SRC   = "corr_img.jpg"
OUT   = "C:/pipeline_out/"
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(0)
if not os.path.exists(SRC):
    c = cv2.imread(CLEAN, cv2.IMREAD_GRAYSCALE)
    cv2.imwrite(SRC, img_as_ubyte(np.clip(
        random_noise(img_as_float(c), mode="gaussian", var=(10 / 255) ** 2, rng=0), 0, 1)))

def w(name, arr):
    cv2.imwrite(OUT + name, np.clip(arr, 0, 255).astype(np.uint8))

# STAGE 1 — IMAGE ACQUISITION
img  = cv2.imread(SRC, cv2.IMREAD_GRAYSCALE)
imgf = img.astype(np.float64); flat = imgf.ravel()
w("s01_input.png", img)
print("STAGE 1  load    : shape=%s dtype=%s" % (img.shape, img.dtype))

# STAGE 2 — SCALE & RANGE
vmin, vmax, vptp = img.min(), img.max(), np.ptp(img)
pcts = np.percentile(flat, [0.1, 1, 99, 99.9]); q1, med_v, q3 = np.percentile(flat, [25, 50, 75])
print("STAGE 2  range   : min=%d max=%d ptp=%d" % (vmin, vmax, vptp))

# STAGE 3 — CENTRAL TENDENCY
mean, median = np.mean(flat), np.median(flat)
tmean = stats.trim_mean(flat, 0.1); mode = float(stats.mode(img.ravel()).mode)
print("STAGE 3  centre  : mean=%.1f median=%.1f trim=%.1f mode=%.0f" % (mean, median, tmean, mode))

# STAGE 4 — SPREAD
std = np.std(flat); iqr = stats.iqr(flat); mad = stats.median_abs_deviation(flat, scale="normal")
print("STAGE 4  spread  : std=%.1f iqr=%.1f MAD=%.1f" % (std, iqr, mad))

# STAGE 5 — OUTLIER IDENTIFICATION
zmap = stats.zscore(flat).reshape(img.shape); outlier_mask = np.abs(zmap) > 3
n_out = int(outlier_mask.sum()); labeled, n_clusters = ndimage.label(outlier_mask)
sizes = (np.sort(ndimage.sum(np.ones_like(labeled), labeled, range(1, n_clusters + 1)))[::-1] if n_clusters else np.array([0]))
mx = int(sizes[0]); w("s05_outlier_mask.png", outlier_mask.astype(np.uint8) * 255)
print("STAGE 5  outlier : |z|>3 -> %d px; largest cluster %d px = crack" % (n_out, mx))

# STAGE 6 — SHAPE
hist = cv2.calcHist([img], [0], None, [256], [0, 256]).ravel().astype(np.int64); centers = np.arange(256) + 0.5
sk, ku = stats.skew(flat), stats.kurtosis(flat)
sk_p = stats.skewtest(flat).pvalue; ku_p = stats.kurtosistest(flat).pvalue
ent = shannon_entropy(img); cdf = np.cumsum(hist) / hist.sum()
print("STAGE 6  shape   : skew=%.2f kurt=%.2f entropy=%.2f" % (sk, ku, ent))

# STAGE 7 — NOISE ID
f01 = img_as_float(img); sigma255 = estimate_sigma(f01, channel_axis=None) * 255
lmean = ndimage.uniform_filter(imgf, 7); lvar = ndimage.generic_filter(imgf, np.var, size=7)
mv_corr = np.corrcoef(lmean.ravel(), lvar.ravel())[0, 1]
resid = imgf - lmean; flat_reg = lvar < np.percentile(lvar, 10); noise = resid[flat_reg]
sigma_flat = stats.median_abs_deviation(noise, scale="normal"); noise_kurt = stats.kurtosis(noise)
nt_p = stats.normaltest(noise).pvalue; zk = (noise - noise.mean()) / noise.std()
ks_p = stats.kstest(zk, "norm").pvalue
cvm_p = stats.cramervonmises(noise, "norm", args=(noise.mean(), noise.std())).pvalue
is_impulse = (hist[0] + hist[255]) / flat.size > 0.005 and vmin == 0 and vmax == 255
model = ("Impulse" if is_impulse else "Signal-dependent" if mv_corr > 0.3 else "Additive Gaussian")
print("STAGE 7  noiseID : sigma_hat=%.1f flat-MAD=%.1f corr=%+.2f => %s" % (sigma255, sigma_flat, mv_corr, model))

# STAGE 8 — OUTLIER REMOVAL (skipped). NB: 'x8' is the noisy pass-through, NOT the clean reference.
x8 = imgf.copy()
print("STAGE 8  outrem  : SKIPPED (flagged pixels are the crack)")

# STAGE 9 — NORMALISE (affine; shown, not used for metrics)
norm01 = (x8 - x8.min()) / (x8.max() - x8.min()); w("s09_normalised.png", norm01 * 255)
print("STAGE 9  normal  : min-max to [0,1] (affine)")

# STAGE 10 — FILTER (native scale)
sig01 = estimate_sigma(f01, channel_axis=None)
denoised = img_as_ubyte(np.clip(denoise_nl_means(f01, patch_size=5, patch_distance=6, h=0.8 * sig01, sigma=sig01, fast_mode=True), 0, 1))
w("s10_denoised.png", denoised)
sig_after = estimate_sigma(img_as_float(denoised), channel_axis=None) * 255
print("STAGE 10 filter  : NL-means -> residual sigma %.1f -> %.2f" % (sigma255, sig_after))

# ══════════════════════════════════════════════════════════════════════════════
# STAGE-METRICS — full-reference, UNBIASED (see bias controls in the header)
# ══════════════════════════════════════════════════════════════════════════════
ref = cv2.imread(CLEAN, cv2.IMREAD_GRAYSCALE)              # TRUE clean reference (native scale)
assert ref.shape == img.shape, "reference and image must be the same size/registration"

def metrics(x):                                            # vs clean ref, native scale
    x = np.clip(x, 0, 255).astype(np.uint8)
    return MSE(ref, x), PSNR(ref, x, data_range=255), SSIM(ref, x, data_range=255)

band = ndimage.binary_dilation(ref < (ref.mean() - ref.std()), iterations=3)   # crack band from CLEAN
def band_ssim(x):
    x = np.clip(x, 0, 255).astype(np.uint8)
    _, smap = SSIM(ref, x, data_range=255, full=True); return float(smap[band].mean())

mn, pn, sn = metrics(img)                                  # noisy baseline
md, pd, sd = metrics(denoised)                             # denoised (native scale)
mse_np = float(np.mean((ref.astype(float) - denoised.astype(float)) ** 2))   # independent MSE check
print("\nSTAGE-METRICS (full-reference vs clean, native scale; PSNR is a function of MSE):")
print("  %-10s %9s %9s %9s %11s" % ("", "MSE", "PSNR", "SSIM", "crack-SSIM"))
print("  %-10s %9.2f %9.2f %9.4f %11.4f" % ("noisy", mn, pn, sn, band_ssim(img)))
print("  %-10s %9.2f %9.2f %9.4f %11.4f" % ("denoised", md, pd, sd, band_ssim(denoised)))
print("  improvement: dPSNR=%+.2f dB  dSSIM=%+.4f  (MSE cross-check: skimage %.2f == numpy %.2f)"
      % (pd - pn, sd - sn, md, mse_np))

# Unbiased EXPECTED performance over K independent noise realisations
K = 10; P, S, Sb = [], [], []
for k in range(K):
    ny = img_as_ubyte(np.clip(random_noise(img_as_float(ref), mode="gaussian", var=(10 / 255) ** 2, rng=k), 0, 1))
    s =   estimate_sigma(img_as_float(ny), channel_axis=None)
    dn = img_as_ubyte(np.clip(denoise_nl_means(img_as_float(ny), patch_size=5, patch_distance=6, h=0.8 * s, sigma=s, fast_mode=True), 0, 1))
    _, p, ss = metrics(dn); P.append(p); S.append(ss); Sb.append(band_ssim(dn))
print("  expected over %d draws: PSNR %.2f±%.2f dB | SSIM %.4f±%.4f | crack-SSIM %.4f±%.4f"
      % (K, np.mean(P), np.std(P), np.mean(S), np.std(S), np.mean(Sb), np.std(Sb)))

# FIGURE
plt.rcParams.update({"font.size": 10})

fig, ax = plt.subplots(3, 6, figsize=(30, 15.5)); ax = ax.ravel(); L = list(string.ascii_lowercase)
def IMG(i, im, t, cmap="gray", vmin=0, vmax=255):
    ax[i].imshow(im, cmap=cmap, vmin=vmin, vmax=vmax); ax[i].set_title("(%s) %s" % (L[i], t), fontsize=10.5, fontweight="bold")
    ax[i].set_xticks([]); ax[i].set_yticks([])
def GT(i, t):
    ax[i].set_title("(%s) %s" % (L[i], t), fontsize=10.5, fontweight="bold"); ax[i].grid(True, alpha=0.3, lw=0.6)
def barlabels(a, bars, fmt="%.1f"):
    for b in bars:
        a.annotate(fmt % b.get_height(), (b.get_x() + b.get_width() / 2, b.get_height()),
                   textcoords="offset points", xytext=(0, 2), ha="center", va="bottom", fontsize=9, fontweight="bold")

IMG(0, img, "Stage 1 · Input image")
ax[1].bar(centers, hist, width=1, color="lightsteelblue"); ax[1].axvline(mode, color="crimson", lw=1.2, ls="--")
GT(1, "Stage 1 · Histogram (calcHist)"); ax[1].set_xlabel("Intensity [0–255]"); ax[1].set_ylabel("Count")
IMG(2, img, "Stage 2 · Range")
bp = ax[3].boxplot(flat, vert=True, whis=(0.1, 99.9), widths=0.5, patch_artist=True); bp["boxes"][0].set_facecolor("lightsteelblue")
for yv, lab in [(vmin, "min %d" % vmin), (med_v, "med %.0f" % med_v), (vmax, "max %d" % vmax)]:
    ax[3].annotate(lab, (1.30, yv), fontsize=8.5, va="center", fontweight="bold")
GT(3, "Stage 2 · Box plot"); ax[3].set_ylabel("Intensity [0–255]"); ax[3].set_xticks([])
IMG(4, img, "Stage 3 · Central tendency")
b3 = ax[5].bar(["mean", "median", "trim", "mode"], [mean, median, tmean, mode], color=["crimson", "navy", "teal", "green"]); barlabels(ax[5], b3)
ax[5].set_ylim(min(mean, median, tmean, mode) - 10, max(mean, median, tmean, mode) + 10); GT(5, "Stage 3 · Estimators"); ax[5].set_ylabel("Intensity")
IMG(6, img, "Stage 4 · Spread")
b4 = ax[7].bar(["std", "IQR", "MAD"], [std, iqr, mad], color=["crimson", "grey", "green"]); barlabels(ax[7], b4)
GT(7, "Stage 4 · Classical vs robust"); ax[7].set_ylabel("Intensity units")
IMG(8, outlier_mask, "Stage 5 · Outlier mask = crack", vmin=0, vmax=1)
top = sizes[:10]; b5 = ax[9].bar(range(1, len(top) + 1), top, color="indianred"); barlabels(ax[9], b5, "%d")
GT(9, "Stage 5 · Cluster sizes (n=%d)" % n_clusters); ax[9].set_xlabel("Rank"); ax[9].set_ylabel("Size [px]")
IMG(10, img, "Stage 6 · Distribution shape")
sub = rng.choice(flat, 2000, replace=False); (osm, osr), (sl, ic, rq) = stats.probplot(sub, dist="norm")
ax[11].plot(osm, osr, ".", ms=3, color="steelblue"); ax[11].plot(osm, sl * osm + ic, "crimson", lw=1.6, label="R²=%.3f" % rq ** 2)
ax[11].legend(fontsize=8); GT(11, "Stage 6 · Q–Q vs Normal (skew %.2f)" % sk); ax[11].set_xlabel("Theoretical [σ]"); ax[11].set_ylabel("Sample")
IMG(12, lvar, "Stage 7 · Local-variance map", cmap="magma", vmin=float(lvar.min()), vmax=float(lvar.max()))
idx = rng.choice(lmean.size, 3000, replace=False); mvx, mvy = lmean.ravel()[idx], lvar.ravel()[idx]
ax[13].plot(mvx, mvy, ".", ms=3, color="steelblue", alpha=.5); sl2, ic2 = np.polyfit(mvx, mvy, 1); xs = np.linspace(mvx.min(), mvx.max(), 50)
ax[13].plot(xs, sl2 * xs + ic2, "crimson", lw=2, label="slope=%.2f" % sl2); ax[13].legend(fontsize=8)
GT(13, "Stage 7 · Mean–variance -> %s" % model); ax[13].set_xlabel("Local mean"); ax[13].set_ylabel("Local variance")
IMG(14, norm01, "Stage 9 · Normalised [0,1]", vmin=0, vmax=1)
xin = np.arange(256); yout = np.clip((xin - x8.min()) / (x8.max() - x8.min()), 0, 1)
ax[15].plot(xin, yout, color="darkorange", lw=2.2); GT(15, "Stage 9 · Transfer function (affine)"); ax[15].set_xlabel("Input [0–255]"); ax[15].set_ylabel("Output [0,1]")
IMG(16, denoised, "Stage 10 · Denoised (NL-means)")
r0 = int(np.unravel_index(np.argmin(ndimage.uniform_filter(imgf, 5)), img.shape)[0])
ax[17].plot(img[r0, :], color="#9ec3e0", lw=1, label="noisy σ=%.1f" % sigma255); ax[17].plot(denoised[r0, :], color="seagreen", lw=2, label="denoised σ=%.1f" % sig_after)
ax[17].legend(fontsize=8); GT(17, "Stage 10 · Profile across crack (row %d)" % r0); ax[17].set_xlabel("Column [px]"); ax[17].set_ylabel("Intensity")

for a in ax:
    a.add_patch(Rectangle((0, 0), 1, 1, transform=a.transAxes, fill=False, edgecolor="#444444", lw=1.6, clip_on=False, zorder=20))
plt.tight_layout(rect=[0.006, 0.022, 0.994, 0.972])
fig.add_artist(Rectangle((0.006, 0.02), 0.988, 0.958, transform=fig.transFigure, fill=False, edgecolor="black", lw=3, zorder=1000))
plt.savefig(OUT + "pipeline_flow.png", dpi=110, bbox_inches="tight"); plt.close()
print("\nSaved outputs to:", os.path.abspath(OUT))

STAGE 1  load    : shape=(227, 227) dtype=uint8
STAGE 2  range   : min=31 max=237 ptp=206
STAGE 3  centre  : mean=166.3 median=169.0 trim=168.1 mode=172
STAGE 4  spread  : std=22.2 iqr=26.0 MAD=19.3
STAGE 5  outlier : |z|>3 -> 622 px; largest cluster 222 px = crack
STAGE 6  shape   : skew=-1.03 kurt=2.24 entropy=6.42
STAGE 7  noiseID : sigma_hat=9.9 flat-MAD=9.0 corr=-0.38 => Additive Gaussian
STAGE 8  outrem  : SKIPPED (flagged pixels are the crack)
STAGE 9  normal  : min-max to [0,1] (affine)
STAGE 10 filter  : NL-means -> residual sigma 9.9 -> 0.83

STAGE-METRICS (full-reference vs clean, native scale; PSNR is a function of MSE):
                   MSE      PSNR      SSIM  crack-SSIM
  noisy         100.38     28.11    0.5746      0.6979
  denoised       16.28     36.01    0.9016      0.9039
  improvement: dPSNR=+7.90 dB  dSSIM=+0.3270  (MSE cross-check: skimage 16.28 == numpy 16.28)
  expected over 10 draws: PSNR 36.03±0.05 dB | SSIM 0.9021±0.0012 | crack-SSIM 0.9032±0.0025

Saved 